# Pipeline 2: Social Media Donation Referrals

## New Dawn Lighthouse Foundation - Explanatory Regression Model

**Business Problem:** The Lighthouse Foundation uses social media to raise awareness about trafficking, share survivor stories, and drive donations. But not all posts are created equal -- some generate significant donation referrals while others receive engagement but no financial conversions. Understanding *which post characteristics* drive donation referrals enables the communications team to optimize their content strategy for maximum fundraising impact.

**Target Variable:** `donation_referrals` - continuous count of donations attributed to a specific social media post.

**Approach:** This is primarily an **explanatory model** (Ch. 9) because the goal is to understand *why* certain posts drive more referrals, not to predict the exact number for a future post. We use:
- **Ch. 6 (Univariate Profiling):** Profile all post-level features.
- **Ch. 7 (Data Preparation):** Handle skewness in engagement metrics, dummy-code categorical variables with reference category dropping.
- **Ch. 8 (Bivariate Analysis):** Examine each feature's relationship with donation_referrals.
- **Ch. 9 (Explanatory Modeling):** Build a statsmodels OLS regression with full diagnostics (residual analysis, Q-Q plots, VIF, Durbin-Watson).
- **Ch. 10 (Predictive Comparison):** Train a sklearn LinearRegression as a predictive benchmark.

The key deliverable is a **coefficient summary table** that tells the communications team exactly which post characteristics (platform, content type, call-to-action style) have the largest impact on donation referrals.

---
## Section 1: Data Loading & Initial Exploration (Ch. 5)

We load the `social_media_posts` table, which contains one row per post with engagement metrics (impressions, reach, likes, shares, etc.) and content attributes (platform, post_type, media_type, content_topic, sentiment_tone, call_to_action_type). The `donation_referrals` column tracks how many donations were directly attributed to each post via referral tracking links.

We also load the `donations` table to cross-reference the `referral_post_id` field and validate our target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

import sys
sys.path.insert(0, '.')
from functions import unistats, bin_categories, skew_correct, missing_fill, clean_outlier, n2n_analysis, n2c_analysis, c2c_analysis, calculate_vif

print('Libraries loaded successfully.')

In [ ]:
# Load data
posts = pd.read_csv('../lighthouse_csv_v7/social_media_posts.csv', parse_dates=['created_at'])
donations = pd.read_csv('../lighthouse_csv_v7/donations.csv')

print(f'Social media posts: {posts.shape}')
print(f'Donations: {donations.shape}')
print(f'\nDonations with referral_post_id: {donations["referral_post_id"].notna().sum()}')

In [ ]:
posts.head(3)

In [ ]:
posts.info()

In [ ]:
# Validate target variable
print(f'Target: donation_referrals')
print(f'  Range: [{posts["donation_referrals"].min()}, {posts["donation_referrals"].max()}]')
print(f'  Mean: {posts["donation_referrals"].mean():.2f}')
print(f'  Median: {posts["donation_referrals"].median():.2f}')
print(f'  Std: {posts["donation_referrals"].std():.2f}')
print(f'  Zero-referral posts: {(posts["donation_referrals"] == 0).sum()} ({(posts["donation_referrals"] == 0).mean():.1%})')

### 1.1 Feature Selection Rationale

We select features based on what the communications team can control when creating a post:

**Categorical Features (to be dummy-coded):**
- `platform` - Facebook, Instagram, YouTube, TikTok, etc. Different platforms have different audience behaviors.
- `post_type` - The format of the post (e.g., image, video, story, reel).
- `media_type` - Type of media attached to the post.
- `content_topic` - What the post is about (advocacy, impact story, fundraising appeal, etc.).
- `sentiment_tone` - Emotional tone of the post content.
- `call_to_action_type` - Type of CTA included (donate, share, learn more, etc.).

**Numeric Features:**
- `impressions` - Total times the post was shown.
- `reach` - Unique users who saw the post.
- `caption_length` - Length of post caption (proxy for content depth).

We intentionally exclude engagement metrics (likes, comments, shares) from the features because they are *outcomes* of the post, not inputs the team controls. Including them would create a post-treatment bias problem: a post that gets many referrals likely also gets high engagement, but the engagement does not *cause* the referrals -- both are effects of the same content quality.

In [ ]:
# Select features for modeling
cat_features = ['platform', 'post_type', 'media_type', 'content_topic', 'sentiment_tone', 'call_to_action_type']
num_features = ['impressions', 'reach', 'caption_length']
target = 'donation_referrals'

# Create working dataset
df = posts[cat_features + num_features + [target]].copy()
print(f'Working dataset shape: {df.shape}')
df.head()

---
## Section 2: Univariate Profiling (Ch. 6)

Chapter 6's univariate profiling helps us understand each variable's distribution before examining relationships. For this regression context, we are especially interested in:
- **Target variable distribution:** If `donation_referrals` is heavily right-skewed (many posts with 0-1 referrals, few with high counts), we may need to transform it or use a different model family.
- **Numeric feature skewness:** Impressions and reach are typically right-skewed in social media data.
- **Categorical cardinality:** We need to identify categories with very few observations that should be binned.

In [ ]:
# Full univariate profile
profile = unistats(df)
profile

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram
axes[0].hist(df[target], bins=30, color='#A2C9E1', edgecolor='white')
axes[0].set_title(f'{target} Distribution\nskew={df[target].skew():.2f}')
axes[0].set_xlabel(target)
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df[target].dropna())
axes[1].set_title(f'{target} Box Plot')

# Q-Q plot
stats.probplot(df[target].dropna(), plot=axes[2])
axes[2].set_title(f'{target} Q-Q Plot')

plt.suptitle('Target Variable Analysis (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(num_features):
    df[col].hist(bins=30, ax=axes[i], color='#91B191', edgecolor='white')
    axes[i].set_title(f'{col}\nskew={df[col].skew():.2f}')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='mean')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label='median')
    axes[i].legend(fontsize=8)
plt.suptitle('Numeric Feature Distributions (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(cat_features):
    ax = axes[i // 3, i % 3]
    df[col].value_counts().plot(kind='bar', ax=ax, color='#E8A87C', edgecolor='white')
    ax.set_title(f'{col} (n={df[col].nunique()})')
    ax.set_ylabel('Count')
    plt.sca(ax)
    plt.xticks(rotation=45, ha='right')
plt.suptitle('Categorical Feature Distributions (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3: Data Preparation & Cleaning (Ch. 7)

Chapter 7 covers the data preparation steps needed before regression modeling. For OLS regression, the key assumptions are:
1. **Linearity:** The relationship between features and target is approximately linear.
2. **Normality of residuals:** Errors should be approximately normally distributed.
3. **Homoscedasticity:** Error variance should be constant across fitted values.
4. **No multicollinearity:** Features should not be highly correlated with each other.

To support these assumptions, we:
- Apply log1p transformation to right-skewed numeric features (stabilizes variance and linearizes relationships).
- Bin rare categories into 'Other' to avoid sparse dummy variables.
- Drop one category per categorical feature (reference coding) to avoid the dummy variable trap.

In [ ]:
# Handle missing values (Ch. 7)
missing_report = df.isnull().sum()
print('Missing Value Report:')
print(missing_report[missing_report > 0])

# Fill numeric with median, categorical with mode
for col in num_features + [target]:
    if df[col].isnull().any():
        df = missing_fill(df, col, strategy='median')
        print(f'  Filled {col} with median')

for col in cat_features:
    if df[col].isnull().any():
        df = missing_fill(df, col, strategy='mode')
        print(f'  Filled {col} with mode')

print(f'\nRemaining missing: {df.isnull().sum().sum()}')

In [ ]:
# Skewness correction for numeric features (Ch. 7)
print('Skewness before correction:')
for col in num_features:
    print(f'  {col}: {df[col].skew():.3f}')

for col in num_features:
    if abs(df[col].skew()) > 1:
        df = skew_correct(df, col, method='log')
        print(f'  -> {col} log-corrected: new skew = {df[col].skew():.3f}')

print(f'\nSkewness after correction:')
for col in num_features:
    print(f'  {col}: {df[col].skew():.3f}')

In [ ]:
# Bin rare categories (Ch. 7)
for col in cat_features:
    before = df[col].nunique()
    df = bin_categories(df, col, threshold=0.03)
    after = df[col].nunique()
    if before != after:
        print(f'{col}: {before} -> {after} categories')
    else:
        print(f'{col}: {before} categories (unchanged)')

In [ ]:
# Create dummy variables (drop_first for reference category - Ch. 9)
df_model = pd.get_dummies(df, columns=cat_features, drop_first=True, dtype=int)
print(f'Modeling dataset shape: {df_model.shape}')
print(f'Columns: {list(df_model.columns)}')

---
## Section 4: Bivariate Analysis (Ch. 8)

Chapter 8 teaches us to examine feature-target relationships before modeling. For regression:
- **N2N (Numeric to Numeric):** Pearson correlation between each numeric feature and `donation_referrals`. Scatterplots reveal the shape of the relationship (linear, curved, no pattern).
- **N2C (Numeric target by Categorical feature):** ANOVA tests whether mean referrals differ significantly across categories (e.g., do Instagram posts generate more referrals than Facebook posts?).

These tests serve as a preliminary feature screen. Features with no significant relationship to the target may not contribute to the regression model.

In [ ]:
# N2N: Numeric features vs donation_referrals (Ch. 8)
for col in num_features:
    result = n2n_analysis(df, col, target)
    print(f'{col} -> {target}: r={result["pearson_r"]:.3f}, p={result["p_value"]:.4f}')

**N2N Interpretation (Ch. 8):** The Pearson correlations reveal the linear relationship strength between each numeric feature and donation referrals. Impressions and reach are expected to show positive correlations -- posts seen by more people naturally have more opportunities for referrals. Caption length may show a weaker or non-linear relationship, as excessively long captions can reduce engagement while very short ones lack persuasive content.

In [ ]:
# N2C: Donation referrals by each categorical feature (Ch. 8)
for col in cat_features:
    result = n2c_analysis(df, target, col)
    print(f'{target} by {col}: F={result["f_statistic"]:.2f}, p={result["p_value"]:.4f}\n')

**N2C Interpretation (Ch. 8):** The ANOVA results reveal which categorical features create statistically significant differences in donation referrals:

- **platform:** Different social media platforms likely show significantly different referral rates. Platforms with built-in donation features (e.g., Facebook Fundraisers) may outperform others.
- **content_topic:** Certain topics (e.g., direct impact stories vs. general advocacy) likely resonate more with potential donors.
- **call_to_action_type:** Direct donation CTAs should outperform awareness-focused CTAs for this outcome.
- **sentiment_tone:** The emotional framing of posts may influence donor motivation.

These insights already provide actionable guidance for the communications team, and the regression model will quantify the exact magnitude of each effect.

---
## Section 5: Model Building (Ch. 9, 10)

### Primary Model: Statsmodels OLS Regression (Ch. 9)

The OLS regression is the centerpiece of this pipeline because our goal is **explanatory**. The coefficient table tells us:
- The **direction** of each feature's effect (positive = increases referrals, negative = decreases).
- The **magnitude** (how many additional referrals for each unit increase).
- The **statistical significance** (p-values tell us which effects are real vs. due to chance).
- The **model fit** (R-squared tells us how much variance in referrals our features explain).

We also perform full **regression diagnostics** as required by Ch. 9:
- **Residual plots:** Check for patterns indicating non-linearity or heteroscedasticity.
- **Q-Q plot:** Assess normality of residuals.
- **VIF (Variance Inflation Factor):** Detect multicollinearity among features.
- **Durbin-Watson:** Test for autocorrelation in residuals.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

# Define X and y
y = df_model[target]
X = df_model.drop(columns=[target])

# Add constant for intercept
X_const = sm.add_constant(X.astype(float))

# Fit OLS model
ols_model = sm.OLS(y, X_const).fit()
print(ols_model.summary())

In [ ]:
# Extract coefficient table with interpretation
coef_table = pd.DataFrame({
    'coefficient': ols_model.params,
    'std_error': ols_model.bse,
    't_statistic': ols_model.tvalues,
    'p_value': ols_model.pvalues,
    'ci_lower': ols_model.conf_int()[0],
    'ci_upper': ols_model.conf_int()[1]
})
coef_table['significant'] = coef_table['p_value'] < 0.05
coef_table = coef_table.sort_values('p_value')

print('OLS Coefficient Table (Ch. 9):')
print(f'R-squared: {ols_model.rsquared:.4f}')
print(f'Adjusted R-squared: {ols_model.rsquared_adj:.4f}')
print(f'F-statistic: {ols_model.fvalue:.2f} (p={ols_model.f_pvalue:.4e})')
coef_table

**Coefficient Interpretation (Ch. 9):**

Each coefficient in the OLS table represents the expected change in `donation_referrals` for a one-unit increase in that feature, holding all other features constant.

For **dummy variables** (categorical features), the coefficient represents the difference in expected referrals compared to the reference category (the dropped first level). For example, if `platform_Instagram` has a coefficient of 2.5, it means Instagram posts generate 2.5 more referrals on average than the reference platform, after controlling for all other features.

For **numeric features** that were log-transformed, the interpretation is: a 1% increase in the original feature corresponds to approximately `coefficient / 100` change in referrals.

The **R-squared** value tells us what proportion of the variance in donation referrals is explained by our model. The **Adjusted R-squared** penalizes for the number of predictors, giving a more honest assessment of model fit.

In [ ]:
# Visualize significant coefficients
sig_coefs = coef_table[(coef_table['significant']) & (coef_table.index != 'const')].sort_values('coefficient')

if len(sig_coefs) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(sig_coefs) * 0.4)))
    colors = ['#27AE60' if x > 0 else '#E74C3C' for x in sig_coefs['coefficient']]
    ax.barh(sig_coefs.index, sig_coefs['coefficient'], color=colors,
            xerr=[sig_coefs['coefficient'] - sig_coefs['ci_lower'], 
                  sig_coefs['ci_upper'] - sig_coefs['coefficient']], capsize=3)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Coefficient (effect on donation_referrals)')
    ax.set_title('Significant OLS Coefficients (Ch. 9)\nGreen = positive effect, Red = negative effect')
    plt.tight_layout()
    plt.show()
else:
    print('No individually significant coefficients at p < 0.05.')

### 5.1 Regression Diagnostics (Ch. 9)

Chapter 9 requires thorough diagnostic checks to validate that OLS assumptions hold. Violations of these assumptions can lead to biased coefficients, incorrect standard errors, and misleading p-values.

1. **Residual vs. Fitted Plot:** Should show random scatter. Patterns indicate non-linearity or omitted variables.
2. **Q-Q Plot of Residuals:** Points should follow the 45-degree line. Deviations in the tails indicate non-normality.
3. **Scale-Location Plot:** Should show random scatter. A funnel shape indicates heteroscedasticity.
4. **VIF (Variance Inflation Factor):** VIF > 5 indicates problematic multicollinearity; VIF > 10 is severe.
5. **Durbin-Watson Statistic:** Values near 2 indicate no autocorrelation; values near 0 or 4 indicate positive or negative autocorrelation.

In [ ]:
# Regression diagnostics (Ch. 9)
residuals = ols_model.resid
fitted = ols_model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residual vs Fitted
axes[0, 0].scatter(fitted, residuals, alpha=0.4, color='#3498DB', s=20)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (check linearity & homoscedasticity)')

# 2. Q-Q Plot
stats.probplot(residuals, plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (check normality of residuals)')

# 3. Scale-Location
axes[1, 0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.4, color='#E74C3C', s=20)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('sqrt(|Residuals|)')
axes[1, 0].set_title('Scale-Location (check homoscedasticity)')

# 4. Residual histogram
axes[1, 1].hist(residuals, bins=30, color='#91B191', edgecolor='white', density=True)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), 'r-', linewidth=2)
axes[1, 1].set_title('Residual Distribution (check normality)')
axes[1, 1].set_xlabel('Residuals')

plt.suptitle('OLS Regression Diagnostics (Ch. 9)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# VIF Analysis (Ch. 9)
vif_results = calculate_vif(X)
print('Variance Inflation Factors (Ch. 9):')
print(f'VIF > 10 (severe multicollinearity): {(vif_results["VIF"] > 10).sum()} features')
print(f'VIF > 5 (moderate multicollinearity): {(vif_results["VIF"] > 5).sum()} features')
vif_results

In [ ]:
# Durbin-Watson test for autocorrelation
dw = durbin_watson(residuals)
print(f'Durbin-Watson Statistic: {dw:.4f}')
if 1.5 < dw < 2.5:
    print('Interpretation: No significant autocorrelation detected (acceptable range: 1.5-2.5).')
elif dw < 1.5:
    print('Interpretation: Positive autocorrelation detected. Consider time-series adjustments or adding lagged features.')
else:
    print('Interpretation: Negative autocorrelation detected. This is less common and may indicate model misspecification.')

**Diagnostics Summary (Ch. 9):**

The diagnostic plots and tests above validate (or flag violations of) the OLS assumptions:

- **Residuals vs Fitted:** If we see random scatter, the linearity assumption is satisfied. A curved pattern would suggest we need polynomial or interaction terms.
- **Q-Q Plot:** Deviations in the tails are common with count data (donation_referrals is bounded at 0) and suggest the normal distribution may not perfectly fit the residuals. For a more rigorous approach, we could use a Poisson or Negative Binomial regression.
- **VIF:** Features with VIF > 5 may need to be removed or combined. Impressions and reach are likely highly correlated (VIF > 10), and we should consider keeping only one.
- **Durbin-Watson:** A value near 2 confirms no autocorrelation, which is expected since our data points are independent posts rather than time-series observations.

In [ ]:
# VIF-based feature selection: remove features with VIF > 10 iteratively (Ch. 9)
X_vif = X.copy()
removed_features = []

while True:
    vif_check = calculate_vif(X_vif)
    max_vif = vif_check['VIF'].max()
    if max_vif > 10 and len(X_vif.columns) > 2:
        worst_feature = vif_check.iloc[0]['feature']
        print(f'Removing {worst_feature} (VIF={max_vif:.1f})')
        removed_features.append(worst_feature)
        X_vif = X_vif.drop(columns=[worst_feature])
    else:
        break

print(f'\nRemoved {len(removed_features)} features due to high VIF: {removed_features}')
print(f'Remaining features: {len(X_vif.columns)}')

In [ ]:
# Refit OLS with VIF-cleaned features
X_vif_const = sm.add_constant(X_vif.astype(float))
ols_model_clean = sm.OLS(y, X_vif_const).fit()

print(f'Original R2: {ols_model.rsquared:.4f} | Adj R2: {ols_model.rsquared_adj:.4f}')
print(f'VIF-clean R2: {ols_model_clean.rsquared:.4f} | Adj R2: {ols_model_clean.rsquared_adj:.4f}')
print(f'\nVIF-cleaned model summary:')
print(ols_model_clean.summary())

### 5.2 Predictive Comparison: sklearn Linear Regression (Ch. 10)

Chapter 10 introduces the predictive modeling framework. While our primary goal is explanatory, we also evaluate the model's predictive performance using a train/test split. This provides a baseline for how well our post features can *forecast* referrals for future posts.

We measure:
- **MAE (Mean Absolute Error):** Average absolute deviation between predicted and actual referrals.
- **RMSE (Root Mean Squared Error):** Penalizes large errors more heavily than MAE.
- **R-squared:** Proportion of variance explained in the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_vif, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit sklearn LinearRegression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print('sklearn LinearRegression - Test Set Performance:')
print(f'  MAE:  {mean_absolute_error(y_test, y_pred_lr):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}')
print(f'  R2:   {r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
# Predicted vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predicted vs actual
axes[0].scatter(y_test, y_pred_lr, alpha=0.5, color='#3498DB')
min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Referrals')
axes[0].set_ylabel('Predicted Referrals')
axes[0].set_title('Predicted vs Actual (Ch. 10)')
axes[0].legend()

# Residual plot
test_residuals = y_test - y_pred_lr
axes[1].scatter(y_pred_lr, test_residuals, alpha=0.5, color='#E74C3C')
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted Referrals')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Test Set Residuals (Ch. 10)')

plt.tight_layout()
plt.show()

---
## Section 6: Deployment - Coefficient Summary for Analytics Dashboard (Ch. 15)

The final deliverable for this explanatory pipeline is a **coefficient summary table** that the communications team can use as a reference when planning social media content. This table translates the statistical model into actionable insights.

Each row shows:
- The feature name (in plain language).
- The coefficient value (expected change in referrals).
- Whether the effect is statistically significant.
- A business interpretation.

This table serves as the bridge between the data science pipeline and operational decision-making at the Lighthouse Foundation.

In [ ]:
# Build deployment-ready coefficient summary
deploy_table = pd.DataFrame({
    'feature': ols_model_clean.params.index,
    'coefficient': ols_model_clean.params.values.round(4),
    'p_value': ols_model_clean.pvalues.values.round(4),
    'significant': (ols_model_clean.pvalues < 0.05).values,
    'direction': ['positive' if c > 0 else 'negative' for c in ols_model_clean.params.values]
})
deploy_table = deploy_table[deploy_table['feature'] != 'const'].sort_values('p_value')

print('Coefficient Summary Table for Analytics Dashboard (Ch. 15):')
print(f'Model R-squared: {ols_model_clean.rsquared:.4f}')
print(f'Model Adj R-squared: {ols_model_clean.rsquared_adj:.4f}')
print(f'Significant features: {deploy_table["significant"].sum()} / {len(deploy_table)}')
deploy_table

In [ ]:
# Save coefficient table for dashboard integration
deploy_table.to_csv('models/social_media_coefficients.csv', index=False)
print('Coefficient table saved to models/social_media_coefficients.csv')

---
## Summary & Business Recommendations

### Key Findings

This pipeline built an explanatory OLS regression model to understand which social media post characteristics drive donation referrals for the Lighthouse Foundation. The analysis followed the IS 455 textbook methodology from univariate profiling (Ch. 6) through deployment (Ch. 15), with particular emphasis on regression diagnostics (Ch. 9).

### Business Recommendations for the Communications Team

1. **Platform Prioritization:** The coefficient table reveals which platforms generate the most referrals. The team should allocate more content creation resources to high-performing platforms while maintaining a presence on others for awareness.

2. **Content Strategy:** The content_topic and sentiment_tone coefficients provide a data-driven content calendar strategy. Posts that feature direct impact stories with specific emotional tones likely outperform generic advocacy content for donation conversion.

3. **CTA Optimization:** The call_to_action_type coefficients tell us exactly which CTA styles convert to donations. The team should A/B test the top-performing CTA types against variations to further optimize.

4. **Reach vs. Quality:** The impressions/reach coefficients quantify how much additional visibility translates to additional donations, helping justify (or question) boosted post budgets.

5. **Caption Length Sweet Spot:** The caption_length coefficient indicates whether longer or shorter captions perform better for donation conversion, guiding content writing guidelines.

### Model Limitations

- **Count data:** `donation_referrals` is a count variable bounded at 0. A Poisson or Negative Binomial regression might be more appropriate for formal inference.
- **Endogeneity:** Some features (especially engagement metrics, which we excluded) are simultaneously determined with referrals.
- **External factors:** Donation behavior is influenced by seasonality, economic conditions, and current events not captured in post-level data.
- **Sample size:** The model's stability should be monitored as more posts are collected.